# 🎌 Anime & Manga Analytics Dashboard
### End-to-End Pipeline: Data Cleaning · EDA · ML Modelling · Visualisations · PostgreSQL Schema


In [ ]:
import warnings, re, io
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error,
                              accuracy_score, classification_report, confusion_matrix)
from sklearn.metrics import ConfusionMatrixDisplay

plt.style.use("dark_background")
sns.set_palette("husl")
pio.templates.default = "plotly_dark"

ANIME_PATH = "/mnt/user-data/uploads/anime_dataset.csv"
MANGA_PATH  = "/mnt/user-data/uploads/manga_dataset.csv"
print("All imports successful")

## 1 · Data Loading & Inspection

In [ ]:
anime_raw = pd.read_csv(ANIME_PATH)
manga_raw = pd.read_csv(MANGA_PATH)
print(f"Anime : {anime_raw.shape[0]:,} rows x {anime_raw.shape[1]} cols")
print(f"Manga : {manga_raw.shape[0]:,} rows x {manga_raw.shape[1]} cols")
print("\nAnime columns:", list(anime_raw.columns))
print("\nManga columns:", list(manga_raw.columns))

In [ ]:
anime_raw.head(3)

In [ ]:
manga_raw.head(3)

In [ ]:
# Missing value heatmap
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Missing-Value % per Column", fontsize=14, fontweight="bold", color="white")
for ax, df, label in zip(axes, [anime_raw, manga_raw], ["Anime", "Manga"]):
    miss = df.isnull().mean().sort_values(ascending=True)
    colors = ["#ff4757" if v>0.5 else "#ffa502" if v>0.2 else "#2ed573" for v in miss.values]
    ax.barh(miss.index, miss.values*100, color=colors, edgecolor="none", height=0.7)
    ax.set_title(label, color="white", fontsize=12)
    ax.set_xlabel("% Missing", color="white")
    ax.tick_params(colors="white", labelsize=8)
    ax.axvline(50, color="#ff4757", lw=1, ls="--", alpha=0.6)
plt.tight_layout()
plt.show()

## 2 · Data Cleaning & Preprocessing

In [ ]:
def parse_list_col(series, sep="|"):
    return series.fillna("").apply(lambda x: [i.strip() for i in x.split(sep) if i.strip()])

def duration_to_min(s):
    if pd.isna(s): return np.nan
    h = re.search(r"(\d+)\s*hr", s)
    m = re.search(r"(\d+)\s*min", s)
    return int(h.group(1))*60 + (int(m.group(1)) if m else 0) if h else (int(m.group(1)) if m else np.nan)

# ANIME CLEANING
anime = anime_raw.copy()
anime["duration_min"] = anime["duration"].apply(duration_to_min)
anime["aired_from"] = pd.to_datetime(anime["aired_from"], errors="coerce")
anime["aired_to"]   = pd.to_datetime(anime["aired_to"],   errors="coerce")
for col in ["score","scored_by","rank","episodes","duration_min"]:
    anime[col] = anime[col].fillna(anime[col].median())
for col in ["type","source","rating","season","studios","genres","themes","demographics"]:
    anime[col] = anime[col].fillna("Unknown")
anime["year"] = anime["year"].fillna(anime["aired_from"].dt.year)
anime["genres_list"]       = parse_list_col(anime_raw["genres"])
anime["themes_list"]       = parse_list_col(anime_raw["themes"])
anime["demographics_list"] = parse_list_col(anime_raw["demographics"])
anime["studios_list"]      = parse_list_col(anime_raw["studios"])
anime["popularity_tier"]   = pd.cut(anime["popularity"], bins=[0,500,2000,8000,np.inf], labels=["Top","High","Mid","Low"])
anime["score_tier"]        = pd.cut(anime["score"], bins=[0,5,6.5,7.5,10], labels=["Poor","Average","Good","Excellent"])

# MANGA CLEANING
manga = manga_raw.copy()
manga["published_from"] = pd.to_datetime(manga["published_from"], errors="coerce")
manga["published_to"]   = pd.to_datetime(manga["published_to"],   errors="coerce")
for col in ["score","scored_by","rank","chapters","volumes"]:
    manga[col] = manga[col].fillna(manga[col].median())
for col in ["type","status","genres","themes","demographics","authors","serializations"]:
    manga[col] = manga[col].fillna("Unknown")
manga["genres_list"]       = parse_list_col(manga_raw["genres"])
manga["themes_list"]       = parse_list_col(manga_raw["themes"])
manga["demographics_list"] = parse_list_col(manga_raw["demographics"])
manga["pub_span_years"] = ((manga["published_to"] - manga["published_from"]).dt.days / 365.25).clip(lower=0)
manga["pub_span_years"] = manga["pub_span_years"].fillna(manga["pub_span_years"].median())
manga["score_tier"]     = pd.cut(manga["score"], bins=[0,5,6.5,7.5,10], labels=["Poor","Average","Good","Excellent"])

print("Anime cleaned:", anime.shape)
print("Manga cleaned:", manga.shape)

## 3 · Exploratory Data Analysis

### 3.1 Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Score Distribution — Anime vs Manga", fontsize=14, fontweight="bold", color="white")
for ax, df, label, color in zip(axes, [anime, manga], ["Anime","Manga"], ["#ff6b9d","#7bed9f"]):
    ax.hist(df["score"], bins=60, color=color, alpha=0.85, edgecolor="none")
    ax.axvline(df["score"].mean(),   color="white",  lw=2, ls="--", label=f"Mean {df['score'].mean():.2f}")
    ax.axvline(df["score"].median(), color="#ffd32a", lw=2, ls=":", label=f"Median {df['score'].median():.2f}")
    ax.set_title(label, color="white"); ax.set_xlabel("Score", color="white")
    ax.tick_params(colors="white"); ax.legend(facecolor="#1e1e2e", labelcolor="white")
plt.tight_layout(); plt.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=anime["score"], name="Anime", opacity=0.7, marker_color="#ff6b9d", nbinsx=60))
fig.add_trace(go.Histogram(x=manga["score"], name="Manga", opacity=0.7, marker_color="#7bed9f", nbinsx=60))
fig.update_layout(barmode="overlay", title="Score Distribution (Interactive — Anime vs Manga)",
                  xaxis_title="Score", yaxis_title="Count")
fig.show()

### 3.2 Anime Type Analysis

In [ ]:
type_stats = anime.groupby("type").agg(
    count=("mal_id","count"), avg_score=("score","mean"),
    avg_members=("members","mean"), avg_episodes=("episodes","mean")
).sort_values("count", ascending=False).reset_index()

colors = px.colors.qualitative.Vivid[:len(type_stats)]
fig = make_subplots(rows=2, cols=2,
    subplot_titles=["Count by Type","Avg Score by Type","Share (Pie)","Avg Episodes"],
    specs=[[{"type":"bar"},{"type":"bar"}],[{"type":"pie"},{"type":"bar"}]])
fig.add_trace(go.Bar(x=type_stats["type"], y=type_stats["count"], marker_color=colors, name="Count"), row=1, col=1)
fig.add_trace(go.Bar(x=type_stats["type"], y=type_stats["avg_score"].round(2), marker_color=colors, name="Score"), row=1, col=2)
fig.add_trace(go.Pie(labels=type_stats["type"], values=type_stats["count"], marker_colors=colors, name="Share"), row=2, col=1)
fig.add_trace(go.Bar(x=type_stats["type"], y=type_stats["avg_episodes"].round(1), marker_color=colors, name="Ep"), row=2, col=2)
fig.update_layout(height=700, title_text="Anime Type Breakdown", showlegend=False)
fig.show()

### 3.3 Genre Analysis

In [ ]:
anime_genre_counts = Counter(g for gl in anime["genres_list"] for g in gl if g)
manga_genre_counts = Counter(g for gl in manga["genres_list"] for g in gl if g)
top_n = 20
a_genres = pd.DataFrame(anime_genre_counts.most_common(top_n), columns=["genre","count"])
m_genres = pd.DataFrame(manga_genre_counts.most_common(top_n), columns=["genre","count"])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(f"Top {top_n} Genres", fontsize=14, fontweight="bold", color="white")
for ax, df, label, cmap_name in zip(axes, [a_genres, m_genres], ["Anime","Manga"], ["RdPu","YlGn"]):
    cmap = plt.cm.get_cmap(cmap_name, len(df))
    bars = ax.barh(df["genre"][::-1], df["count"][::-1],
                   color=[cmap(i) for i in range(len(df))], edgecolor="none")
    ax.set_title(label, color="white", fontsize=12)
    ax.set_xlabel("Count", color="white"); ax.tick_params(colors="white", labelsize=9)
    for bar in bars:
        ax.text(bar.get_width()+10, bar.get_y()+bar.get_height()/2,
                f"{bar.get_width():,.0f}", va="center", color="white", fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
def genre_score_df(df, genre_col="genres_list", label_col="score"):
    rows = []
    for _, row in df[[genre_col, label_col]].iterrows():
        for g in row[genre_col]:
            if g: rows.append({"genre": g, "score": row[label_col]})
    tmp = pd.DataFrame(rows)
    return tmp.groupby("genre").agg(count=("score","count"), avg_score=("score","mean")).reset_index()

ag = genre_score_df(anime)
ag = ag[ag["count"] > 100]
fig = px.scatter(ag, x="count", y="avg_score", size="count", color="avg_score",
                 text="genre", color_continuous_scale="Plasma",
                 title="Anime Genres — Popularity vs Avg Score",
                 labels={"count":"# Titles","avg_score":"Avg Score"})
fig.update_traces(textposition="top center", textfont_size=9)
fig.update_layout(height=600)
fig.show()

### 3.4 Score Over Time

In [ ]:
yearly = (anime[anime["year"].between(1980,2024)]
          .groupby("year")
          .agg(avg_score=("score","mean"), count=("mal_id","count"))
          .reset_index())
fig = make_subplots(specs=[[{"secondary_y":True}]])
fig.add_trace(go.Scatter(x=yearly["year"], y=yearly["avg_score"],
    mode="lines+markers", name="Avg Score", line=dict(color="#ff6b9d", width=2)), secondary_y=False)
fig.add_trace(go.Bar(x=yearly["year"], y=yearly["count"], name="# Titles",
    opacity=0.4, marker_color="#7bed9f"), secondary_y=True)
fig.update_layout(title="Anime: Avg Score & Title Count Over Time", xaxis_title="Year")
fig.update_yaxes(title_text="Avg Score", secondary_y=False)
fig.update_yaxes(title_text="# Titles", secondary_y=True)
fig.show()

### 3.5 Correlation Heatmaps

In [ ]:
num_cols_a = ["score","scored_by","rank","popularity","members","favorites","episodes","duration_min","year"]
corr_a = anime[num_cols_a].corr()
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_a, dtype=bool))
sns.heatmap(corr_a, annot=True, fmt=".2f", cmap="coolwarm", mask=mask, ax=ax,
            linewidths=0.3, annot_kws={"size":8})
ax.set_title("Anime Feature Correlation Matrix", color="white", fontsize=13)
ax.tick_params(colors="white")
plt.tight_layout(); plt.show()

In [ ]:
num_cols_m = ["score","scored_by","rank","popularity","members","favorites","chapters","volumes","pub_span_years"]
corr_m = manga[num_cols_m].corr()
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_m, dtype=bool))
sns.heatmap(corr_m, annot=True, fmt=".2f", cmap="PiYG", mask=mask, ax=ax,
            linewidths=0.3, annot_kws={"size":8})
ax.set_title("Manga Feature Correlation Matrix", color="white", fontsize=13)
ax.tick_params(colors="white")
plt.tight_layout(); plt.show()

### 3.6 Top Studios (Treemap) & Demographics

In [ ]:
studio_counts = Counter(s for sl in anime["studios_list"] for s in sl if s and s != "Unknown")
top_studios = pd.DataFrame(studio_counts.most_common(25), columns=["studio","count"])
fig = px.treemap(top_studios, path=["studio"], values="count", color="count",
                 color_continuous_scale="Magma", title="Top 25 Anime Studios by # Titles")
fig.update_layout(height=500); fig.show()

In [ ]:
demo_a = anime[~anime["demographics"].isin(["Unknown",""])]["demographics"].value_counts()
demo_m = manga[~manga["demographics"].isin(["Unknown",""])]["demographics"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Demographics Distribution", fontsize=13, fontweight="bold", color="white")
for ax, data, label in zip(axes, [demo_a, demo_m], ["Anime","Manga"]):
    wedges, texts, autotexts = ax.pie(data.values, labels=data.index, autopct="%1.1f%%",
        pctdistance=0.8, startangle=90, colors=plt.cm.Set2.colors[:len(data)])
    for t in texts + autotexts: t.set_color("white"); t.set_fontsize(9)
    ax.set_title(label, color="white")
plt.tight_layout(); plt.show()

### 3.7 Source Material & Rating (Anime)

In [ ]:
source_score = anime.groupby("source").agg(count=("mal_id","count"), avg_score=("score","mean")).reset_index()
source_score = source_score[source_score["count"] > 50].sort_values("avg_score", ascending=False)
fig = px.bar(source_score, x="source", y="avg_score", color="count",
             color_continuous_scale="Turbo",
             title="Avg Score by Source Material",
             labels={"avg_score":"Avg Score","count":"# Titles","source":"Source"})
fig.update_layout(xaxis_tickangle=-30); fig.show()

### 3.8 Manga Type & Chapters vs Score

In [ ]:
manga_type = manga.groupby("type").agg(count=("mal_id","count"), avg_score=("score","mean")).reset_index().sort_values("count", ascending=False)
fig = make_subplots(rows=1, cols=2, specs=[[{"type":"pie"},{"type":"bar"}]],
                    subplot_titles=["Manga Type Distribution","Avg Score by Type"])
fig.add_trace(go.Pie(labels=manga_type["type"], values=manga_type["count"], hole=0.45, name="Count"), row=1, col=1)
fig.add_trace(go.Bar(x=manga_type["type"], y=manga_type["avg_score"].round(2),
                     marker_color=px.colors.qualitative.Bold, name="Score"), row=1, col=2)
fig.update_layout(height=450, title_text="Manga Type Analysis"); fig.show()

In [ ]:
sample_m = manga[manga["chapters"] < 1000].sample(min(5000, len(manga)), random_state=42)
fig = px.scatter(sample_m, x="chapters", y="score", color="type", size="members",
                 hover_data=["title"], opacity=0.6, title="Chapters vs Score (Manga)",
                 color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(height=500); fig.show()

### 3.9 Season & Release Calendar (Anime)

In [ ]:
season_data = anime[anime["season"] != "Unknown"].copy()
season_stats = season_data.groupby(["season","year"]).agg(count=("mal_id","count")).reset_index()
season_stats = season_stats[season_stats["year"].between(2010,2024)]
fig = px.density_heatmap(season_stats, x="year", y="season", z="count",
                          color_continuous_scale="Viridis",
                          title="Anime Releases per Season & Year (2010-2024)")
fig.update_layout(height=400); fig.show()

In [ ]:
season_order = ["spring","summer","fall","winter"]
season_score = (anime[anime["season"].isin(season_order)]
                .groupby("season")["score"]
                .apply(list).reindex(season_order))
fig, ax = plt.subplots(figsize=(12, 5))
parts = ax.violinplot([season_score[s] for s in season_order], showmeans=True, showmedians=True)
for pc, c in zip(parts["bodies"], ["#ff6b9d","#ffd32a","#ff6348","#7bed9f"]):
    pc.set_facecolor(c); pc.set_alpha(0.75)
ax.set_xticks(range(1,5)); ax.set_xticklabels(season_order, color="white", fontsize=11)
ax.set_title("Score Distribution by Season (Violin)", color="white", fontsize=13)
ax.set_ylabel("Score", color="white"); ax.tick_params(colors="white")
plt.tight_layout(); plt.show()

### 3.10 Top-Rated Titles (Bar Race Style)

In [ ]:
top_anime = anime.nlargest(20,"score")[["title","score","members","type","year"]].reset_index(drop=True)
fig = px.bar(top_anime, x="score", y="title", color="type", orientation="h",
             title="Top 20 Highest Rated Anime", text="score",
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig.update_traces(texttemplate="%{text:.2f}", textposition="inside")
fig.update_layout(height=600, yaxis=dict(autorange="reversed")); fig.show()

In [ ]:
top_manga = manga.nlargest(20,"score")[["title","score","members","type"]].reset_index(drop=True)
fig = px.bar(top_manga, x="score", y="title", color="type", orientation="h",
             title="Top 20 Highest Rated Manga", text="score",
             color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_traces(texttemplate="%{text:.2f}", textposition="inside")
fig.update_layout(height=600, yaxis=dict(autorange="reversed")); fig.show()

### 3.11 Cumulative Releases, Radar & Box Plots

In [ ]:
anime_yr = anime[anime["year"].between(1960,2024)]["year"].value_counts().sort_index().cumsum()
manga_yr  = manga["published_from"].dt.year.dropna()
manga_yr  = manga_yr[manga_yr.between(1960,2024)].value_counts().sort_index().cumsum()
fig = go.Figure()
fig.add_trace(go.Scatter(x=anime_yr.index, y=anime_yr.values, name="Anime", fill="tozeroy", line=dict(color="#ff6b9d", width=2)))
fig.add_trace(go.Scatter(x=manga_yr.index, y=manga_yr.values, name="Manga", fill="tozeroy", line=dict(color="#7bed9f", width=2)))
fig.update_layout(title="Cumulative Releases Over Time", xaxis_title="Year", yaxis_title="Cumulative Count", height=450)
fig.show()

In [ ]:
top8 = [g for g,_ in Counter(g for gl in anime["genres_list"] for g in gl).most_common(8)]
def genre_avg_score(df, top_genres):
    rows = []
    for _, row in df[["genres_list","score"]].iterrows():
        for g in row["genres_list"]:
            if g in top_genres: rows.append({"genre":g, "score":row["score"]})
    tmp = pd.DataFrame(rows)
    return tmp.groupby("genre")["score"].mean().reindex(top_genres)

anime_radar = genre_avg_score(anime, top8)
manga_radar = genre_avg_score(manga, top8)
fig = go.Figure()
fig.add_trace(go.Scatterpolar(r=anime_radar.values, theta=top8, fill="toself", name="Anime", line_color="#ff6b9d"))
fig.add_trace(go.Scatterpolar(r=manga_radar.values, theta=top8, fill="toself", name="Manga", line_color="#7bed9f"))
fig.update_layout(title="Avg Score by Genre (Radar Chart)", height=500,
                  polar=dict(radialaxis=dict(range=[5.5, 8.5])))
fig.show()

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Anime Score by Type","Manga Score by Type"])
for t in anime["type"].dropna().unique():
    fig.add_trace(go.Box(y=anime[anime["type"]==t]["score"], name=t, boxmean=True), row=1, col=1)
for t in manga["type"].dropna().unique():
    fig.add_trace(go.Box(y=manga[manga["type"]==t]["score"], name=t, boxmean=True), row=1, col=2)
fig.update_layout(height=500, showlegend=False, title_text="Score Distribution by Type")
fig.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Favorites vs Members (log-log)", fontsize=13, fontweight="bold", color="white")
for ax, df, label, color in zip(axes, [anime, manga], ["Anime","Manga"], ["#ff6b9d","#7bed9f"]):
    ax.scatter(np.log1p(df["members"]), np.log1p(df["favorites"]), s=5, alpha=0.3, color=color)
    ax.set_xlabel("log(Members)", color="white"); ax.set_ylabel("log(Favorites)", color="white")
    ax.set_title(label, color="white"); ax.tick_params(colors="white")
plt.tight_layout(); plt.show()

## 4 · Feature Engineering

In [ ]:
anime_ml = anime.copy()
for col in ["members","favorites","scored_by"]:
    anime_ml[f"log_{col}"] = np.log1p(anime_ml[col])
le = LabelEncoder()
for col in ["type","source","season","rating"]:
    anime_ml[f"{col}_enc"] = le.fit_transform(anime_ml[col].astype(str))
anime_ml["is_ongoing"] = anime_ml["airing"].astype(int)
top_genres_a = [g for g,_ in Counter(g for gl in anime_ml["genres_list"] for g in gl).most_common(10)]
for g in top_genres_a:
    anime_ml[f"genre_{g.replace(' ','_')}"] = anime_ml["genres_list"].apply(lambda gl: int(g in gl))

feature_cols_anime = (
    ["log_members","log_favorites","log_scored_by","episodes","duration_min","year",
     "type_enc","source_enc","season_enc","rating_enc","is_ongoing"]
    + [f"genre_{g.replace(' ','_')}" for g in top_genres_a]
)
print("Anime features:", len(feature_cols_anime))

manga_ml = manga.copy()
for col in ["members","favorites","scored_by"]:
    manga_ml[f"log_{col}"] = np.log1p(manga_ml[col])
le2 = LabelEncoder()
for col in ["type","status","demographics"]:
    manga_ml[f"{col}_enc"] = le2.fit_transform(manga_ml[col].astype(str))
top_genres_m = [g for g,_ in Counter(g for gl in manga_ml["genres_list"] for g in gl).most_common(10)]
for g in top_genres_m:
    manga_ml[f"genre_{g.replace(' ','_')}"] = manga_ml["genres_list"].apply(lambda gl: int(g in gl))
feature_cols_manga = (
    ["log_members","log_favorites","log_scored_by","chapters","volumes","pub_span_years",
     "type_enc","status_enc","demographics_enc"]
    + [f"genre_{g.replace(' ','_')}" for g in top_genres_m]
)
print("Manga features:", len(feature_cols_manga))

## 5 · Machine Learning Models

### 5.1 Anime Score Regression

In [ ]:
anime_model_df = anime_ml[feature_cols_anime + ["score"]].dropna()
X_a = anime_model_df[feature_cols_anime]; y_a = anime_model_df["score"]
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_a, y_a, test_size=0.2, random_state=42)
scaler_a = StandardScaler()
X_train_a_sc = scaler_a.fit_transform(X_train_a); X_test_a_sc = scaler_a.transform(X_test_a)

models_reg_a = {
    "LinearRegression" : LinearRegression(),
    "Ridge"            : Ridge(alpha=1.0),
    "RandomForest"     : RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "GradientBoosting" : GradientBoostingRegressor(n_estimators=150, max_depth=5, random_state=42),
}
results_reg_a = {}
for name, model in models_reg_a.items():
    use_sc = name in ("LinearRegression","Ridge")
    model.fit(X_train_a_sc if use_sc else X_train_a.values, y_train_a)
    preds = model.predict(X_test_a_sc if use_sc else X_test_a.values)
    results_reg_a[name] = {"RMSE": np.sqrt(mean_squared_error(y_test_a, preds)),
                            "MAE" : mean_absolute_error(y_test_a, preds),
                            "R2"  : r2_score(y_test_a, preds), "preds": preds}
    print(f"{name:20s}  RMSE={results_reg_a[name]['RMSE']:.4f}  MAE={results_reg_a[name]['MAE']:.4f}  R2={results_reg_a[name]['R2']:.4f}")
best_model_a = max(results_reg_a, key=lambda k: results_reg_a[k]["R2"])
print(f"\nBest model: {best_model_a}")

In [ ]:
reg_df = pd.DataFrame({k: {m: v for m,v in v.items() if m!="preds"}
                        for k,v in results_reg_a.items()}).T.reset_index()
reg_df.columns = ["Model","RMSE","MAE","R2"]
fig = make_subplots(rows=1, cols=3, subplot_titles=["RMSE (lower better)","MAE (lower better)","R2 (higher better)"])
for i, metric in enumerate(["RMSE","MAE","R2"], 1):
    fig.add_trace(go.Bar(x=reg_df["Model"], y=reg_df[metric],
        marker_color=px.colors.qualitative.Safe, text=reg_df[metric].round(4),
        textposition="outside", showlegend=False), row=1, col=i)
fig.update_layout(height=420, title_text="Anime Score Regression — Model Comparison"); fig.show()

In [ ]:
best_preds_a = results_reg_a[best_model_a]["preds"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Anime Score Regression — {best_model_a}", fontsize=13, fontweight="bold", color="white")
axes[0].scatter(y_test_a, best_preds_a, alpha=0.3, s=10, color="#ff6b9d")
mn, mx = y_test_a.min(), y_test_a.max()
axes[0].plot([mn,mx],[mn,mx],"w--", lw=2)
axes[0].set_xlabel("Actual Score", color="white"); axes[0].set_ylabel("Predicted", color="white")
axes[0].set_title("Actual vs Predicted", color="white"); axes[0].tick_params(colors="white")
residuals = np.array(y_test_a) - best_preds_a
axes[1].hist(residuals, bins=50, color="#7bed9f", edgecolor="none", alpha=0.85)
axes[1].axvline(0, color="white", lw=2, ls="--")
axes[1].set_xlabel("Residual", color="white"); axes[1].set_ylabel("Count", color="white")
axes[1].set_title("Residuals Distribution", color="white"); axes[1].tick_params(colors="white")
plt.tight_layout(); plt.show()

### 5.2 Feature Importance

In [ ]:
rf_model_a = models_reg_a["RandomForest"]
fi = pd.Series(rf_model_a.feature_importances_, index=feature_cols_anime).sort_values(ascending=False)
fig = px.bar(x=fi.index, y=fi.values, color=fi.values, color_continuous_scale="Plasma",
             title="Feature Importance — Anime Score (RandomForest)",
             labels={"x":"Feature","y":"Importance"})
fig.update_layout(xaxis_tickangle=-40); fig.show()

### 5.3 Score Tier Classification — Manga

In [ ]:
manga_clf_df = manga_ml[feature_cols_manga + ["score_tier"]].dropna()
X_m = manga_clf_df[feature_cols_manga]; y_m = manga_clf_df["score_tier"]
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_m, y_m, test_size=0.2, random_state=42, stratify=y_m)
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_m, y_train_m); y_pred_m = rf_clf.predict(X_test_m)
print("Classification Report — Manga Score Tier")
print(classification_report(y_test_m, y_pred_m))

In [ ]:
cm = confusion_matrix(y_test_m, y_pred_m, labels=rf_clf.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_clf.classes_)
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix — Manga Score Tier", color="white", fontsize=12)
ax.tick_params(colors="white"); ax.xaxis.label.set_color("white"); ax.yaxis.label.set_color("white")
for text in ax.texts: text.set_color("black")
plt.tight_layout(); plt.show()

### 5.4 Manga Score Regression

In [ ]:
manga_reg_df = manga_ml[feature_cols_manga + ["score"]].dropna()
X_mr = manga_reg_df[feature_cols_manga]; y_mr = manga_reg_df["score"]
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(X_mr, y_mr, test_size=0.2, random_state=42)
rf_manga = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_manga.fit(X_tr_m, y_tr_m); preds_m = rf_manga.predict(X_te_m)
print(f"Manga RF Regression  RMSE={np.sqrt(mean_squared_error(y_te_m, preds_m)):.4f}  "
      f"MAE={mean_absolute_error(y_te_m, preds_m):.4f}  R2={r2_score(y_te_m, preds_m):.4f}")
fi_m = pd.Series(rf_manga.feature_importances_, index=feature_cols_manga).sort_values(ascending=False)
fig = px.bar(x=fi_m.index, y=fi_m.values, color=fi_m.values, color_continuous_scale="Magma",
             title="Feature Importance — Manga Score RF Regressor",
             labels={"x":"Feature","y":"Importance"})
fig.update_layout(xaxis_tickangle=-40); fig.show()

## 6 · Real-Time Analytical Dashboard Plots

In [ ]:
# KPI summary
kpis = {
    "Total Anime"       : f"{len(anime):,}",
    "Total Manga"       : f"{len(manga):,}",
    "Avg Anime Score"   : f"{anime['score'].mean():.2f}",
    "Avg Manga Score"   : f"{manga['score'].mean():.2f}",
    "Top Anime"         : anime.nlargest(1,'score')['title'].values[0][:30],
    "Top Manga"         : manga.nlargest(1,'score')['title'].values[0][:30],
    "Anime Genres"      : str(len(anime_genre_counts)),
    "Manga Genres"      : str(len(manga_genre_counts)),
    "Top Studio"        : Counter(s for sl in anime["studios_list"] for s in sl if s and s!="Unknown").most_common(1)[0][0],
}
fig = go.Figure()
for i, (k, v) in enumerate(kpis.items()):
    col = i % 3; row = i // 3
    fig.add_annotation(x=col/2.5, y=1-row*0.3,
        text=f"<b>{k}</b><br><span style='font-size:20px;color:#7bed9f'>{v}</span>",
        showarrow=False, font=dict(size=13, color="white"),
        align="center", xref="paper", yref="paper",
        bgcolor="#1e1e2e", bordercolor="#7bed9f", borderpad=8, borderwidth=1)
fig.update_layout(height=370, paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
                  title="Dashboard KPIs", xaxis_visible=False, yaxis_visible=False)
fig.show()

In [ ]:
# Sunburst hierarchy
sun_df = []
for _, row in anime[["type","demographics","genres_list"]].iterrows():
    demo = row["demographics"] if row["demographics"] not in ["Unknown",""] else "Unspecified"
    for g in (row["genres_list"][:1] if row["genres_list"] else ["Other"]):
        sun_df.append({"type":row["type"], "demo":demo, "genre":g})
sun_df = pd.DataFrame(sun_df)
sun_agg = sun_df.groupby(["type","demo","genre"]).size().reset_index(name="count")
sun_agg = sun_agg[sun_agg["count"] > 5]
fig = px.sunburst(sun_agg, path=["type","demo","genre"], values="count",
                  color="count", color_continuous_scale="RdBu",
                  title="Anime Hierarchy: Type > Demographics > Primary Genre")
fig.update_layout(height=600); fig.show()

In [ ]:
# Interactive scatter: members vs score
sample_a = anime.sample(min(8000, len(anime)), random_state=42)
fig = px.scatter(sample_a, x="members", y="score", color="type", size="favorites",
                 hover_data=["title","year","genres"], opacity=0.55, log_x=True,
                 title="Anime: Audience Size vs Score",
                 color_discrete_sequence=px.colors.qualitative.G10)
fig.update_layout(height=550); fig.show()

In [ ]:
sample_mg = manga.sample(min(8000, len(manga)), random_state=42)
fig = px.scatter(sample_mg, x="members", y="score", color="type", size="favorites",
                 hover_data=["title"], opacity=0.55, log_x=True,
                 title="Manga: Audience Size vs Score",
                 color_discrete_sequence=px.colors.qualitative.Alphabet)
fig.update_layout(height=550); fig.show()

## 7 · PostgreSQL Schema & Insert Scripts

In [ ]:
SQL_SCHEMA = "-- ============================================================\n-- Anime & Manga Analytics — PostgreSQL Schema\n-- PostgreSQL 14+\n-- ============================================================\nCREATE EXTENSION IF NOT EXISTS pg_trgm;\n\n-- Dimension tables\nCREATE TABLE IF NOT EXISTS genres      (genre_id SERIAL PRIMARY KEY, name TEXT UNIQUE NOT NULL);\nCREATE TABLE IF NOT EXISTS themes      (theme_id SERIAL PRIMARY KEY, name TEXT UNIQUE NOT NULL);\nCREATE TABLE IF NOT EXISTS demographics(demo_id  SERIAL PRIMARY KEY, name TEXT UNIQUE NOT NULL);\nCREATE TABLE IF NOT EXISTS studios     (studio_id SERIAL PRIMARY KEY, name TEXT UNIQUE NOT NULL);\nCREATE TABLE IF NOT EXISTS authors     (author_id SERIAL PRIMARY KEY, name TEXT UNIQUE NOT NULL);\n\n-- Main tables\nCREATE TABLE IF NOT EXISTS anime (\n    mal_id INT PRIMARY KEY, title TEXT NOT NULL, title_english TEXT, title_japanese TEXT,\n    type TEXT, source TEXT, episodes INT, status TEXT, airing BOOLEAN,\n    aired_from DATE, aired_to DATE, duration_min NUMERIC(6,1), rating TEXT,\n    score NUMERIC(4,2), scored_by INT, rank INT, popularity INT, members INT,\n    favorites INT, season TEXT, year SMALLINT, synopsis TEXT, image_url TEXT,\n    created_at TIMESTAMP DEFAULT NOW()\n);\nCREATE TABLE IF NOT EXISTS anime_genres      (anime_id INT REFERENCES anime(mal_id) ON DELETE CASCADE, genre_id INT REFERENCES genres(genre_id), PRIMARY KEY(anime_id,genre_id));\nCREATE TABLE IF NOT EXISTS anime_themes      (anime_id INT REFERENCES anime(mal_id) ON DELETE CASCADE, theme_id INT REFERENCES themes(theme_id), PRIMARY KEY(anime_id,theme_id));\nCREATE TABLE IF NOT EXISTS anime_demographics(anime_id INT REFERENCES anime(mal_id) ON DELETE CASCADE, demo_id  INT REFERENCES demographics(demo_id),  PRIMARY KEY(anime_id,demo_id));\nCREATE TABLE IF NOT EXISTS anime_studios     (anime_id INT REFERENCES anime(mal_id) ON DELETE CASCADE, studio_id INT REFERENCES studios(studio_id),   PRIMARY KEY(anime_id,studio_id));\n\nCREATE TABLE IF NOT EXISTS manga (\n    mal_id INT PRIMARY KEY, title TEXT NOT NULL, title_english TEXT, title_japanese TEXT,\n    type TEXT, chapters INT, volumes INT, status TEXT, publishing BOOLEAN,\n    published_from DATE, published_to DATE, score NUMERIC(4,2), scored_by INT,\n    rank INT, popularity INT, members INT, favorites INT, synopsis TEXT,\n    image_url TEXT, created_at TIMESTAMP DEFAULT NOW()\n);\nCREATE TABLE IF NOT EXISTS manga_genres      (manga_id INT REFERENCES manga(mal_id) ON DELETE CASCADE, genre_id INT REFERENCES genres(genre_id), PRIMARY KEY(manga_id,genre_id));\nCREATE TABLE IF NOT EXISTS manga_themes      (manga_id INT REFERENCES manga(mal_id) ON DELETE CASCADE, theme_id INT REFERENCES themes(theme_id), PRIMARY KEY(manga_id,theme_id));\nCREATE TABLE IF NOT EXISTS manga_demographics(manga_id INT REFERENCES manga(mal_id) ON DELETE CASCADE, demo_id  INT REFERENCES demographics(demo_id),  PRIMARY KEY(manga_id,demo_id));\nCREATE TABLE IF NOT EXISTS manga_authors     (manga_id INT REFERENCES manga(mal_id) ON DELETE CASCADE, author_id INT REFERENCES authors(author_id),   PRIMARY KEY(manga_id,author_id));\n\n-- Indexes\nCREATE INDEX IF NOT EXISTS idx_anime_score      ON anime(score DESC);\nCREATE INDEX IF NOT EXISTS idx_anime_year       ON anime(year);\nCREATE INDEX IF NOT EXISTS idx_anime_type       ON anime(type);\nCREATE INDEX IF NOT EXISTS idx_anime_popularity ON anime(popularity);\nCREATE INDEX IF NOT EXISTS idx_manga_score      ON manga(score DESC);\nCREATE INDEX IF NOT EXISTS idx_manga_type       ON manga(type);\nCREATE INDEX IF NOT EXISTS idx_manga_popularity ON manga(popularity);\nCREATE INDEX IF NOT EXISTS idx_anime_title_trgm ON anime USING GIN(title gin_trgm_ops);\nCREATE INDEX IF NOT EXISTS idx_manga_title_trgm ON manga USING GIN(title gin_trgm_ops);\n\n-- Dashboard views\nCREATE OR REPLACE VIEW v_anime_dashboard AS\nSELECT a.mal_id,a.title,a.type,a.year,a.score,a.members,a.favorites,a.popularity,a.rank,\n       a.season,a.rating,a.episodes,\n       string_agg(DISTINCT g.name,', ') FILTER (WHERE g.name IS NOT NULL) AS genres,\n       string_agg(DISTINCT s.name,', ') FILTER (WHERE s.name IS NOT NULL) AS studios\nFROM anime a\nLEFT JOIN anime_genres ag ON ag.anime_id=a.mal_id LEFT JOIN genres g ON g.genre_id=ag.genre_id\nLEFT JOIN anime_studios ast ON ast.anime_id=a.mal_id LEFT JOIN studios s ON s.studio_id=ast.studio_id\nGROUP BY a.mal_id;\n\nCREATE OR REPLACE VIEW v_manga_dashboard AS\nSELECT m.mal_id,m.title,m.type,m.status,m.score,m.members,m.favorites,m.popularity,m.rank,\n       m.chapters,m.volumes,\n       string_agg(DISTINCT g.name,', ') FILTER (WHERE g.name IS NOT NULL) AS genres,\n       string_agg(DISTINCT au.name,', ') FILTER (WHERE au.name IS NOT NULL) AS authors\nFROM manga m\nLEFT JOIN manga_genres mg ON mg.manga_id=m.mal_id LEFT JOIN genres g ON g.genre_id=mg.genre_id\nLEFT JOIN manga_authors ma ON ma.manga_id=m.mal_id LEFT JOIN authors au ON au.author_id=ma.author_id\nGROUP BY m.mal_id;\n\nCREATE OR REPLACE VIEW v_genre_stats AS\nSELECT g.name AS genre,\n       COUNT(DISTINCT ag.anime_id) AS anime_count,\n       COUNT(DISTINCT mg.manga_id) AS manga_count,\n       ROUND(AVG(a.score)::NUMERIC,2) AS avg_anime_score,\n       ROUND(AVG(m.score)::NUMERIC,2) AS avg_manga_score\nFROM genres g\nLEFT JOIN anime_genres ag ON ag.genre_id=g.genre_id LEFT JOIN anime a ON a.mal_id=ag.anime_id\nLEFT JOIN manga_genres mg ON mg.genre_id=g.genre_id LEFT JOIN manga m ON m.mal_id=mg.manga_id\nGROUP BY g.name ORDER BY (COUNT(DISTINCT ag.anime_id)+COUNT(DISTINCT mg.manga_id)) DESC;\n"
print("PostgreSQL Schema:")
print(SQL_SCHEMA[:3000])
print("...  [truncated for display — full schema written to disk]")

In [ ]:
# Helper functions for SQL generation
def escape_sql(s):
    if s is None or (isinstance(s, float) and np.isnan(s)): return "NULL"
    return "'" + str(s).replace("'","''") + "'"

def to_sql_val(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "NULL"
    if isinstance(v, bool): return "TRUE" if v else "FALSE"
    if isinstance(v, (int, np.integer)): return str(int(v))
    if isinstance(v, (float, np.floating)): return f"{v:.4f}"
    return escape_sql(str(v))

# Anime inserts (sample 500)
cols_a = ["mal_id","title","title_english","title_japanese","type","source",
          "episodes","status","airing","aired_from","aired_to","duration_min",
          "rating","score","scored_by","rank","popularity","members","favorites",
          "season","year","synopsis","image_url"]
buf = io.StringIO()
buf.write("-- Anime INSERT sample (500 rows)\nBEGIN;\n")
for _, row in anime.head(500).iterrows():
    vals = []
    for c in cols_a:
        v = row.get(c)
        if c in ("aired_from","aired_to"): vals.append(escape_sql(str(v.date()) if pd.notna(v) else None))
        else: vals.append(to_sql_val(v))
    buf.write(f"INSERT INTO anime ({','.join(cols_a)}) VALUES ({','.join(vals)}) ON CONFLICT (mal_id) DO NOTHING;\n")
buf.write("COMMIT;\n")
with open("/tmp/anime_inserts.sql","w") as f: f.write(buf.getvalue())
print(f"Wrote /tmp/anime_inserts.sql  ({len(buf.getvalue())//1024} KB)")

In [ ]:
# Manga inserts (sample 500)
cols_m = ["mal_id","title","title_english","title_japanese","type","chapters","volumes",
          "status","publishing","published_from","published_to","score","scored_by",
          "rank","popularity","members","favorites","synopsis","image_url"]
buf2 = io.StringIO()
buf2.write("-- Manga INSERT sample (500 rows)\nBEGIN;\n")
for _, row in manga.head(500).iterrows():
    vals = []
    for c in cols_m:
        v = row.get(c)
        if c in ("published_from","published_to"): vals.append(escape_sql(str(v.date()) if pd.notna(v) else None))
        else: vals.append(to_sql_val(v))
    buf2.write(f"INSERT INTO manga ({','.join(cols_m)}) VALUES ({','.join(vals)}) ON CONFLICT (mal_id) DO NOTHING;\n")
buf2.write("COMMIT;\n")
with open("/tmp/manga_inserts.sql","w") as f: f.write(buf2.getvalue())
print(f"Wrote /tmp/manga_inserts.sql  ({len(buf2.getvalue())//1024} KB)")

# Save schema
with open("/tmp/schema.sql","w") as f: f.write(SQL_SCHEMA)
print("Wrote /tmp/schema.sql")
print()
print("Usage:")
print("  psql -U <user> -d <db> -f schema.sql")
print("  psql -U <user> -d <db> -f anime_inserts.sql")
print("  psql -U <user> -d <db> -f manga_inserts.sql")

## 8 · Summary

In [ ]:
print("="*55)
print("  PIPELINE SUMMARY")
print("="*55)
print(f"  Anime records processed   : {len(anime):>10,}")
print(f"  Manga records processed   : {len(manga):>10,}")
print(f"  Avg anime score           : {anime['score'].mean():>10.2f}")
print(f"  Avg manga score           : {manga['score'].mean():>10.2f}")
print(f"  Best regression model     : {best_model_a:>10}")
print(f"  Best R2 score             : {results_reg_a[best_model_a]['R2']:>10.4f}")
print(f"  Best RMSE                 : {results_reg_a[best_model_a]['RMSE']:>10.4f}")
print(f"  Unique anime genres       : {len(anime_genre_counts):>10}")
print(f"  Unique manga genres       : {len(manga_genre_counts):>10}")
print("="*55)